# External Tool Guardrails with Pramagent

<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/pramagent_tool_guardrails.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook shows how to place an external control layer in front of a LlamaIndex tool. LlamaIndex remains responsible for agent orchestration and tool selection; Pramagent validates the proposed tool call before the side effect runs.

The example uses a refund tool because payments are a useful boundary case: the model may propose the right tool, but the application still needs deterministic policy, human approval, and an audit trail before execution.


## Install dependencies

This example does not require a model provider key. It exercises the tool boundary directly, which is the same boundary a LlamaIndex agent reaches after selecting a tool.


In [ ]:
%pip install -U llama-index-core pramagent


In [ ]:
from llama_index.core.tools import FunctionTool

from pramagent import Pramagent, Verdict
from pramagent.layers import ToolGuardLayer, ToolPolicy
from pramagent.layers.tool_guard import SideEffect


## Define a side-effecting tool

In a production agent this function would call a payment, ticketing, email, or database API. Here we keep the side effect visible by appending to an in-memory list. If Pramagent holds or blocks the action, the list should stay empty.


In [ ]:
side_effects = []


def issue_refund(order_id: str, amount_usd: float, destination_account: str) -> dict:
    """Issue a refund to a customer account."""
    record = {
        "order_id": order_id,
        "amount_usd": amount_usd,
        "destination_account": destination_account,
    }
    side_effects.append(record)
    return {"status": "executed", **record}


## Add deterministic policy outside the model

The policy below does three things before the refund function can execute:

1. validates arguments with JSON Schema,
2. restricts execution to the `finance-demo` tenant and `refund` action label, and
3. escalates valid refund attempts for human approval instead of executing automatically.


In [ ]:
guard = ToolGuardLayer(
    policies=[
        ToolPolicy(
            name="issue_refund",
            side_effect=SideEffect.PAYMENT,
            action=Verdict.ESCALATE,
            allowed_tenants={"finance-demo"},
            allowed_actions={"refund"},
            schema={
                "type": "object",
                "required": ["order_id", "amount_usd", "destination_account"],
                "properties": {
                    "order_id": {"type": "string", "pattern": "^ord-[0-9]+$"},
                    "amount_usd": {"type": "number", "minimum": 0.01, "maximum": 5000},
                    "destination_account": {"type": "string", "pattern": "^acct-[0-9]{6,}$"},
                },
                "additionalProperties": False,
            },
        )
    ]
)

armor = Pramagent(tool_guard=guard)


## Wrap the LlamaIndex tool

The wrapper performs the Pramagent decision first. A block or escalation returns a structured tool result to the agent without running the side effect. Only an `ALLOW` decision calls the underlying function.


In [ ]:
def guarded_issue_refund(
    order_id: str,
    amount_usd: float,
    destination_account: str,
    tenant_id: str = "finance-demo",
    session_id: str = "notebook-demo",
) -> dict:
    proposed_args = {
        "order_id": order_id,
        "amount_usd": amount_usd,
        "destination_account": destination_account,
    }
    decision = armor.validate_tool(
        "issue_refund",
        proposed_args,
        tenant_id=tenant_id,
        session_id=session_id,
        action_label="refund",
    )

    if decision.verdict == Verdict.BLOCK:
        return {
            "status": "blocked",
            "reason": decision.reason,
            "executed": False,
        }

    if decision.verdict == Verdict.ESCALATE:
        return {
            "status": "held_for_human_approval",
            "reason": decision.reason or "policy requires approval",
            "executed": False,
        }

    result = issue_refund(order_id, amount_usd, destination_account)
    return {**result, "executed": True}


refund_tool = FunctionTool.from_defaults(
    fn=guarded_issue_refund,
    name="issue_refund",
    description="Request a customer refund. Valid requests are held for human approval.",
)


## Valid action: held, not executed

The arguments are valid, but the payment side effect is configured to require human approval. The tool returns `held_for_human_approval`, and the underlying refund function never runs.


In [ ]:
held = refund_tool.call(
    order_id="ord-1001",
    amount_usd=1200.00,
    destination_account="acct-123456",
)

print(held.raw_output)
print("side_effects:", side_effects)


## Invalid action: blocked before execution

This request exceeds the schema maximum. It is blocked deterministically before the side-effecting function can run.


In [ ]:
blocked = refund_tool.call(
    order_id="ord-1002",
    amount_usd=9000.00,
    destination_account="acct-123456",
)

print(blocked.raw_output)
print("side_effects:", side_effects)


## Use the guarded tool with an agent

You can pass `refund_tool` to a LlamaIndex agent just like any other `FunctionTool`. LlamaIndex still handles the agent loop and tool selection; Pramagent handles the execution boundary.


In [ ]:
from llama_index.core.agent.workflow import FunctionAgent

agent = FunctionAgent(
    tools=[refund_tool],
    system_prompt=(
        "You help support teams process refund requests. "
        "Use the issue_refund tool when a refund is requested."
    ),
)

# After configuring your preferred LLM, run for example:
# response = await agent.run("Refund order ord-1001 to acct-123456 for $1200.")
# print(response)


## Why this pattern is useful

This keeps the trust boundary outside the model. The model may suggest a tool call, but deterministic policy decides whether the call executes, is held for approval, or is blocked. The same pattern can be applied to email, database writes, account changes, procurement actions, or any other tool with real side effects.
